In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# ===============================
# CONFIGURATION
# ===============================

SERVICE_LEVEL = 0.95
Z = norm.ppf(SERVICE_LEVEL)

# ===============================
# LOAD FILE
# ===============================

df = pd.read_excel("indent_vs_actual_wide_analysis.xlsx")

# ===============================
# IDENTIFY DAILY INDENT COLUMNS ONLY
# ===============================

indent_cols = [
    col for col in df.columns
    if "Indent" in col
    and "Total" not in col
    and "Deviation" not in col
]

print("Indent columns used:")
print(indent_cols)

# ===============================
# CALCULATE MEAN & STD ON INDENT
# ===============================

df["Mean_Indent"] = df[indent_cols].mean(axis=1)
df["Std_Indent"] = df[indent_cols].std(axis=1)

df["Std_Indent"] = df["Std_Indent"].fillna(0)

# ===============================
# CONFIDENCE INTERVAL
# ===============================

df["Indent_CI_Lower"] = df["Mean_Indent"] - Z * df["Std_Indent"]
df["Indent_CI_Upper"] = df["Mean_Indent"] + Z * df["Std_Indent"]

df["Indent_CI_Lower"] = df["Indent_CI_Lower"].apply(lambda x: max(0, x))

# ===============================
# SAVE
# ===============================

df.to_excel("indent_confidence_band_clean.xlsx", index=False)

print("Indent confidence band calculated correctly.")


In [ ]:
import pandas as pd

# =====================================
# LOAD FILES
# =====================================

# Load indent confidence interval file
ci_df = pd.read_excel("indent_confidence_band_clean.xlsx")

# Load updated actual file
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# MERGE 12th & 13th ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH FEB AGAINST INDENT CI
# =====================================

df["12th_Inside_Indent_CI"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper"])
)

# =====================================
# VALIDATE 13TH FEB AGAINST INDENT CI
# =====================================

df["13th_Inside_Indent_CI"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper"])
)

# =====================================
# HIT RATE CALCULATION
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI"].mean() * 100

print("12th Feb Hit Rate (Indent CI):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent CI):", round(hit_rate_13, 2), "%")

# Save validation file
df.to_excel("Indent_CI_validation_12_13.xlsx", index=False)


In [ ]:
import pandas as pd

# =====================================
# LOAD INDENT CI FILE
# =====================================

df = pd.read_excel("indent_confidence_band_clean.xlsx")

# =====================================
# RECOMPUTE INDENT CI USING ±2σ
# =====================================

Z = 2  # ±2 sigma

df["Indent_CI_Lower_2sigma"] = df["Mean_Indent"] - Z * df["Std_Indent"]
df["Indent_CI_Upper_2sigma"] = df["Mean_Indent"] + Z * df["Std_Indent"]

df["Indent_CI_Lower_2sigma"] = df["Indent_CI_Lower_2sigma"].clip(lower=0)

# =====================================
# VALIDATE 12TH FEB
# =====================================

df["12th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH FEB
# =====================================

df["13th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# HIT RATE
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (Indent ±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent ±2σ):", round(hit_rate_13, 2), "%")

# Save
df.to_excel("Indent_CI_2sigma_validation.xlsx", index=False)


In [ ]:
import pandas as pd

# =====================================
# LOAD FILES
# =====================================

ci_df = pd.read_excel("confidence_band_analysis.xlsx")
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# RECOMPUTE CI USING ±2σ
# =====================================

ci_df["CI_Lower_2sigma"] = ci_df["Mean_Actual"] - 2 * ci_df["Std_Actual"]
ci_df["CI_Upper_2sigma"] = ci_df["Mean_Actual"] + 2 * ci_df["Std_Actual"]

ci_df["CI_Lower_2sigma"] = ci_df["CI_Lower_2sigma"].clip(lower=0)

# =====================================
# MERGE 12th & 13th ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH FEB (±2σ)
# =====================================

df["12th_Inside_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH FEB (±2σ)
# =====================================

df["13th_Inside_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

# =====================================
# HIT RATE CALCULATION
# =====================================

hit_rate_12 = df["12th_Inside_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (±2σ):", round(hit_rate_13, 2), "%")

# Save validation file
df.to_excel("CI_validation_12_13_2sigma.xlsx", index=False)


In [ ]:
import pandas as pd

# =====================================
# LOAD INDENT CI FILE
# =====================================

ci_df = pd.read_excel("indent_confidence_band_clean.xlsx")
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# COMPUTE ±2σ ON INDENT
# =====================================

ci_df["Indent_CI_Lower_2sigma"] = ci_df["Mean_Indent"] - 2 * ci_df["Std_Indent"]
ci_df["Indent_CI_Upper_2sigma"] = ci_df["Mean_Indent"] + 2 * ci_df["Std_Indent"]

ci_df["Indent_CI_Lower_2sigma"] = ci_df["Indent_CI_Lower_2sigma"].clip(lower=0)

# =====================================
# MERGE 12TH & 13TH ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH (Indent ±2σ)
# =====================================

df["12th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH (Indent ±2σ)
# =====================================

df["13th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# HIT RATE
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (Indent ±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent ±2σ):", round(hit_rate_13, 2), "%")

# Save
df.to_excel("Indent_CI_2sigma_validation.xlsx", index=False)
